In [12]:
import sys
import subprocess
from bs4 import BeautifulSoup

In [13]:
name_map = {"Adidas": "Adidas+AG",
        "Nike": "Nike+Inc",
        "Puma": "Puma+SE",
        "Lululemon": "Lululemon+Athletica+Inc",
        "H & M": "H+&+M+Hennes+&+Mauritz+AB",
        "JD Sports": "JD+Sports+Fashion+PLC",
        "Gap": "Gap+Inc",
        "Burberry": "Burberry+Group+PLC",
        "Columbia": "Columbia+Sportswear+Co",
        "Boohoo": "Boohoo+Group+PLC",
        "Under Armour": "Under+Armour+Inc"
 }

companies = name_map.keys()

In [14]:
site = "https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data"

In [15]:
def get_html(url, selector=None, wait=False):

    cmd = [sys.executable, "webscrape.py", "--url", url]

    if selector:
        cmd += ["--selector", selector]

    if wait:
        cmd += ["--wait"]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("webscrape.py failed")

    with open("page.html", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    return soup



In [ ]:
ESG_data = {}

def build_ESG_data(ESG_data, company_name):
    full_site_link = site + "?esg=" + name_map[company_name]
    print(full_site_link)

    soup = get_html(
        full_site_link,
        selector="div.EsgRatings.EsgRatings--rating",
        wait=True
    )

    ESG_container = soup.select_one("div.EsgRatings.EsgRatings--rating")

    industry_div = soup.select_one("div.EsgRatings.EsgRatings--rank p")
    industry = industry_div.text.replace("Out of ", "").replace(" Companies.", "").strip()

    categories = ESG_container.find_all("div", class_="EsgRatings EsgRatings--category")

    ESG_data[company_name] = {"industry": industry}

    for category in categories:
        cat_name = category.find("h5").text.strip()
        ESG_data[company_name][cat_name] = {}

        sub_cats = category.find_all("div", class_="EsgRatings EsgRatings--item")

        for sub_cat in sub_cats:
            label = sub_cat.select_one("div.EsgRatings--label")
            score_tag = sub_cat.find("b")

            if not label or not score_tag:
                continue

            sub_cat_name = label.text.strip()
            score = score_tag.text.strip()

            ESG_data[company_name][cat_name][sub_cat_name] = score

    return ESG_data


for company in companies:
    try:
        ESG_data = build_ESG_data(ESG_data, company)
    except Exception as e:
        print(f"Failed for {company}: {e}")
        ESG_data[company] = {"error": str(e)}

print(ESG_data)

https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Adidas+AG
Failed for Adidas: [Errno 2] No such file or directory: 'page.html'
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Nike+Inc
Failed for Nike: [Errno 2] No such file or directory: 'page.html'
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Puma+SE
Failed for Puma: [Errno 2] No such file or directory: 'page.html'
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=Lululemon+Athletica+Inc
Failed for Lululemon: [Errno 2] No such file or directory: 'page.html'
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=H+&+M+Hennes+&+Mauritz+AB
Failed for H & M: [Errno 2] No such file or directory: 'page.html'
https://www.lseg.com/en/data-analytics/sustainable-finance/sustainability-ratings-and-data?esg=JD+Sports+Fashi

In [16]:
import pandas as pd

rows = []

for company, categories in ESG_data.items():
    industry = categories.get("industry")

    for category, metrics in categories.items():
        if category == "industry":
            continue

        for metric, score in metrics.items():
            rows.append({
                "company": company,
                "industry": industry,
                "category": category,
                "metric": metric,
                "score": score
            })

df = pd.DataFrame(rows)

AttributeError: 'str' object has no attribute 'items'

In [ ]:
print(df)

          company            industry       category  \
0          Adidas  Textiles & Apparel  Environmental   
1          Adidas  Textiles & Apparel  Environmental   
2          Adidas  Textiles & Apparel  Environmental   
3          Adidas  Textiles & Apparel  Environmental   
4          Adidas  Textiles & Apparel  Environmental   
..            ...                 ...            ...   
160  Under Armour  Textiles & Apparel     Governance   
161  Under Armour  Textiles & Apparel     Governance   
162  Under Armour  Textiles & Apparel     Governance   
163  Under Armour  Textiles & Apparel     Governance   
164  Under Armour  Textiles & Apparel     Governance   

                            metric score  
0                    Environmental   2.4  
1               Climate Transition     4  
2            Energy & Resource Use     3  
3                     Biodiversity     1  
4                        Water Use     1  
..                             ...   ...  
160                     Go

In [ ]:
df.to_csv("esg_scores.csv", index=False)